In [ ]:
!pip install -q sentence-transformers faiss-cpu pymupdf google-generativeai beautifulsoup4 requests

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI Studio API Key: ")

Enter your Google AI Studio API Key: ··········


In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

Saving SASWAT RESUME.pdf to SASWAT RESUME (1).pdf
Uploaded: SASWAT RESUME (1).pdf


In [ ]:
import fitz  # PyMuPDF

def load_resume(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = load_resume(pdf_path)
print("\nResume Loaded Successfully!")


Resume Loaded Successfully!


In [ ]:
import re

def preprocess_resume(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\+?\d[\d\s\-]{8,}', '', text)
    return text

resume_text = preprocess_resume(resume_text)

In [ ]:
def create_chunks(text):
    sections = re.split(r'\n+', text)
    chunks = []

    for sec in sections:
        sec = sec.strip()
        if len(sec) > 40:
            chunks.append(sec)

    return chunks

chunks = create_chunks(resume_text)
print("Total Chunks:", len(chunks))

Total Chunks: 20


In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks):
    return embed_model.encode(chunks)

embeddings = embed_chunks(chunks)
print("Embeddings Created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Created


In [ ]:
import faiss
import numpy as np

def build_vector_store(embeddings):
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    return index

index = build_vector_store(embeddings)
print("Vector DB Ready")


Vector DB Ready


In [ ]:
def query_vector_store(index, chunks, query, k=5):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [chunks[i] for i in indices[0]]


In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_job_description(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [ ]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)


Extracting Job Skills...
Job Skills: ```python
[]
```


In [ ]:
def compare_skills(resume_skills, job_skills):
    prompt = f"""
    Compare these:

    Resume Skills:
    {resume_skills}

    Job Skills:
    {job_skills}

    Return ONLY:

    Matching Skills: []
    Missing Skills: []
    Match Percentage: %
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
def final_decision(result):
    prompt = f"""
    Based on:

    {result}

    If match percentage >= 70% → SELECTED
    Else → NOT SELECTED

    Output ONLY:
    Decision: SELECTED or NOT SELECTED
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

job_url = input("\nEnter Job URL: ")

print("\nScraping Job Description...")
job_text = scrape_job_description(job_url)

relevant_chunks = query_vector_store(index, chunks, "skills experience technologies", k=5)
filtered_resume_text = " ".join(relevant_chunks)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



Enter Job URL: https://www.naukri.com/job-listings-software-engineer-paramarsh-informatics-delhi-ncr-0-to-5-years-200226014559?src=jobsearchDesk&sid=17743367950208436&xp=3&px=1&nignbevent_src=jobsearchDeskGNB

Scraping Job Description...


In [ ]:
print("\nExtracting Resume Skills...")
resume_skills = extract_skills(filtered_resume_text)
print("Resume Skills:", resume_skills)


Extracting Resume Skills...
Resume Skills: ```python
[
    "user-centered design",
    "wireframing",
    "prototyping",
    "usability testing",
    "web development",
    "backend programming",
    "process optimization",
    "writing clean code"
]
```


In [ ]:
print("\nPlease paste the Job Description here (Press Enter twice to stop):")
job_lines = []
while True:
    line = input()
    if line == "":
        break
    job_lines.append(line)

job_text = "\n".join(job_lines)
print("\nJob Description captured.")


Please paste the Job Description here (Press Enter twice to stop):
Develop and maintain web applications using Java, HTML, CSS, and JavaScript 	•	Work with backend technologies like Node.js / Express and MySQL 	•	Design and implement database structures and queries 	•	Collaborate with team members to build scalable and efficient systems 	•	Debug, test, and optimize application performance 	•	Integrate APIs and third-party services (e.g., authentication, book APIs) 	•	Participate in project development such as Airline Reservation System and Bookique


Job Description captured.


In [ ]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)


Extracting Job Skills...
Job Skills: ```python
[
    "Java",
    "HTML",
    "CSS",
    "JavaScript",
    "Node.js",
    "Express",
    "MySQL",
    "database structures design",
    "database queries implementation",
    "Debugging",
    "Testing",
    "application performance optimization",
    "API integration",
    "third-party services integration"
]
```


In [ ]:
print("\nComparing Skills...")
comparison = compare_skills(resume_skills, job_skills)
print("\n", comparison)

print("\nFinal Decision...")
decision = final_decision(comparison)
print(decision)


Comparing Skills...

 Matching Skills: [
    "Java",
    "HTML",
    "CSS",
    "JavaScript",
    "Node.js",
    "Express",
    "MySQL",
    "database structures design",
    "database queries implementation",
    "Testing",
    "application performance optimization",
    "API integration",
    "third-party services integration"
]
Missing Skills: [
    "Debugging"
]
Match Percentage: 92.86%

Final Decision...
Decision: SELECTED
